# 🗺️ Energy Poverty Mapping – Bangladesh MEPI Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iqbal1999-tiger/energy-poverty-mapping/blob/main/energy_poverty_mapping_colab.ipynb)

This notebook walks you through the full **Multidimensional Energy Poverty Index (MEPI)** workflow using the [energy-poverty-mapping](https://github.com/iqbal1999-tiger/energy-poverty-mapping) repository.

**Sections:**
1. [Setup & Installation](#setup) – Mount Drive, clone repo, install dependencies
2. [Quick Start Examples](#quickstart) – MEPI analysis, maps, visualizations, reports
3. [Data Upload & Processing](#data-upload) – Upload your own CSV and calculate MEPI
4. [Output & Download](#output) – Display results and download files
5. [Custom Configuration](#custom-config) – Custom weights, zone analysis, sensitivity analysis

---
> **Tip:** Run cells in order from top to bottom.  
> Use **Runtime → Run all** to execute the entire notebook at once.

---
## 1. Setup & Installation <a id='setup'></a>

This section:
- Optionally mounts your Google Drive (useful for saving outputs permanently)
- Clones the GitHub repository
- Installs all required Python dependencies

In [ ]:
# ── Optional: Mount Google Drive ────────────────────────────────────────────
# Skip this cell if you don't want to save outputs to Google Drive.

MOUNT_DRIVE = False  # Set to True to mount your Google Drive

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Google Drive mounted at /content/drive')
else:
    print('ℹ️  Skipping Google Drive mount. Outputs will be saved in /content/.')

In [ ]:
# ── Clone the repository ────────────────────────────────────────────────────
import os

REPO_URL = 'https://github.com/iqbal1999-tiger/energy-poverty-mapping.git'
REPO_DIR = '/content/energy-poverty-mapping'

if os.path.isdir(REPO_DIR):
    print(f'Repository already cloned at {REPO_DIR}. Pulling latest changes …')
    !git -C {REPO_DIR} pull --quiet
else:
    print(f'Cloning repository to {REPO_DIR} …')
    !git clone --quiet {REPO_URL} {REPO_DIR}

# Change into the repository directory
os.chdir(REPO_DIR)
print(f'✅ Working directory: {os.getcwd()}')

In [ ]:
# ── Install all dependencies ─────────────────────────────────────────────────
print('Installing dependencies from requirements.txt …')
!pip install -q -r requirements.txt
print('✅ All dependencies installed.')

# Verify key packages
import importlib
required = ['numpy', 'pandas', 'matplotlib', 'seaborn', 'scipy',
            'plotly', 'geopandas', 'folium', 'reportlab', 'docx']
missing = []
for pkg in required:
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f'⚠️  Missing packages: {missing}. Re-run the install cell.')
else:
    print('✅ All required packages are importable.')

---
## 2. Quick Start Examples <a id='quickstart'></a>

Run ready-made example scripts that demonstrate the core functionality of the project.  
Each subsection uses the bundled `sample_data.csv` file.

### 2a. Basic MEPI Analysis

Loads `sample_data.csv`, validates the data, calculates MEPI scores, prints a summary, and exports results to `output/`.

In [ ]:
print('Running example MEPI analysis …')
!python example_mepi_analysis.py
print('\n✅ MEPI analysis complete.')

### 2b. Generate Spatial Maps

Creates choropleth and regional maps of MEPI scores across Bangladesh upazilas.

In [ ]:
print('Generating spatial maps …')
!python example_spatial_maps.py
print('\n✅ Spatial maps generated.')

### 2c. Generate Visualizations

Produces bar charts, heatmaps, scatter plots, and statistical visualizations saved to `output/`.

In [ ]:
print('Generating visualizations …')
!python example_generate_visualizations.py
print('\n✅ Visualizations generated.')

### 2d. Generate Full Report

Generates a comprehensive PDF + DOCX report with all analysis results and charts.

In [ ]:
print('Generating full report (PDF + DOCX) …')
!python generate_full_report.py
print('\n✅ Full report generated.')

### 2e. List Generated Output Files

In [ ]:
import os
import glob

# Common output directories used by the project scripts
output_dirs = [
    'output',
    os.path.expanduser('~/energy_poverty_reports'),
    os.path.expanduser('~/spatial_maps_png'),
    os.path.expanduser('~/map_outputs_energy_poverty'),
    'map_outputs',
]

for d in output_dirs:
    if os.path.isdir(d):
        files = glob.glob(os.path.join(d, '**', '*'), recursive=True)
        files = [f for f in files if os.path.isfile(f)]
        if files:
            print(f'\n📁 {d}/')
            for f in sorted(files):
                size_kb = os.path.getsize(f) / 1024
                print(f'   {os.path.relpath(f, d):60s}  {size_kb:6.1f} KB')
        else:
            print(f'📁 {d}/ (empty)')
    else:
        print(f'📁 {d}/ (not created yet)')

### 2f. Display Visualizations Inline

Display the charts that were generated inside the notebook.

In [ ]:
import glob
import os
from IPython.display import display, Image

# Find PNG files in the output directories
png_files = []
for pattern in ['output/**/*.png', 'output/*.png',
                os.path.expanduser('~/spatial_maps_png/**/*.png'),
                'map_outputs/**/*.png']:
    png_files.extend(glob.glob(pattern, recursive=True))

if png_files:
    print(f'Found {len(png_files)} PNG file(s). Displaying up to 5 …\n')
    for png in sorted(png_files)[:5]:
        print(f'📊 {png}')
        display(Image(filename=png, width=700))
else:
    print('No PNG files found yet. Run the visualisation scripts first (Section 2c).')

---
## 3. Data Upload & Processing <a id='data-upload'></a>

Upload your own CSV file containing energy access survey data, validate it, and calculate MEPI scores.

### Expected CSV Column Format

Your CSV must include the following columns:

| Column | Description |
|--------|-------------|
| `upazila_id` | Unique upazila identifier |
| `upazila_name` | Upazila name |
| `district` | District name |
| `division` | Division name |
| `electricity_access_rate` | % households with electricity access |
| `clean_cooking_fuel_rate` | % households using clean cooking fuel |
| `grid_connection_rate` | % households connected to national grid |
| `hours_supply_per_day` | Average hours of electricity supply/day |
| `outage_frequency_per_month` | Average outages per month |
| `outage_duration_hours` | Average outage duration (hours) |
| `energy_consumption_kwh` | Monthly household energy consumption (kWh) |
| `lighting_hours_per_day` | Average lighting hours/day |
| `appliance_ownership_index` | Appliance ownership index (0–1) |
| `voltage_fluctuation_rate` | Voltage fluctuations per week |
| `energy_satisfaction_score` | Satisfaction score (1–5) |
| `indoor_air_quality_index` | Air quality index (higher = better) |
| `energy_expenditure_share` | % income spent on energy |
| `energy_cost_per_kwh` | Cost per kWh (BDT) |
| `subsidy_access_rate` | % eligible households with subsidy access |

In [ ]:
# ── Upload a custom CSV file ─────────────────────────────────────────────────
# Run this cell to open a file picker and upload your data.
# Skip this cell if you want to use the bundled sample_data.csv instead.

USE_SAMPLE_DATA = True   # Set to False to upload your own CSV

if USE_SAMPLE_DATA:
    DATA_FILE = 'sample_data.csv'
    print(f'ℹ️  Using bundled sample data: {DATA_FILE}')
else:
    from google.colab import files
    print('📂 Please select your CSV file in the dialog below …')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file uploaded. Re-run this cell and select a CSV file.')
    DATA_FILE = list(uploaded.keys())[0]
    print(f'✅ Uploaded: {DATA_FILE}')

In [ ]:
# ── Load and validate the data ───────────────────────────────────────────────
import sys
import os

# Ensure the repository root is on the Python path
REPO_DIR = '/content/energy-poverty-mapping'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

import pandas as pd
from data_utils import load_data, validate_data, handle_missing_values, data_summary

print(f'Loading data from: {DATA_FILE}')
df_raw = load_data(DATA_FILE)
print(f'  Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')

print('\nValidating data …')
df_validated = validate_data(df_raw)

print('Handling missing values …')
df_clean = handle_missing_values(df_validated)

print('\n── Data Summary ──')
data_summary(df_clean)

print('\n── First 5 rows ──')
display(df_clean.head())

In [ ]:
# ── Calculate MEPI scores ────────────────────────────────────────────────────
from mepi_calculator import MEPICalculator
from config import DEFAULT_WEIGHTS

calc = MEPICalculator(weights=DEFAULT_WEIGHTS)
mepi_results = calc.calculate(df_clean)

print(f'✅ MEPI calculated for {len(mepi_results)} upazilas')
print(f'   Mean MEPI score : {mepi_results["mepi_score"].mean():.4f}')
print(f'   Min  MEPI score : {mepi_results["mepi_score"].min():.4f}')
print(f'   Max  MEPI score : {mepi_results["mepi_score"].max():.4f}')

# Show top-10 most energy-poor upazilas
display_cols = ['upazila_name', 'district', 'mepi_score', 'poverty_category',
                'Availability_score', 'Reliability_score', 'Adequacy_score',
                'Quality_score', 'Affordability_score']

print('\n── Top 10 most energy-poor upazilas ──')
display(mepi_results[display_cols].sort_values('mepi_score', ascending=False).head(10).round(4))

---
## 4. Output & Download <a id='output'></a>

Export results to CSV/Excel, display summary statistics, and download files to your local machine.

In [ ]:
# ── Export results to CSV and Excel ─────────────────────────────────────────
import os
from analysis import export_results, summarise_results, print_summary

os.makedirs('output', exist_ok=True)

# Export to both formats
export_results(mepi_results, 'output/mepi_results.xlsx')
mepi_results.to_csv('output/mepi_results.csv', index=False)

print('✅ Results exported:')
print('   output/mepi_results.csv')
print('   output/mepi_results.xlsx')

# Print text summary
print('\n── Summary Statistics ──')
summary = summarise_results(mepi_results)
print_summary(summary)

In [ ]:
# ── Poverty category distribution ────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')   # use non-interactive backend in Colab
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

category_counts = mepi_results['poverty_category'].value_counts()
category_pct    = (category_counts / len(mepi_results) * 100).round(1)

print('Poverty Category Distribution:')
for cat, cnt in category_counts.items():
    pct = category_pct[cat]
    print(f'  {cat:20s}: {cnt:3d} upazilas ({pct:.1f}%)')

# Pie chart
colors = {'Non-Poor': '#2ecc71', 'Moderately Poor': '#f39c12', 'Severely Poor': '#e74c3c'}
fig, ax = plt.subplots(figsize=(7, 5))
wedge_colors = [colors.get(c, '#95a5a6') for c in category_counts.index]
ax.pie(category_counts.values, labels=category_counts.index,
       colors=wedge_colors, autopct='%1.1f%%', startangle=90)
ax.set_title('MEPI Poverty Category Distribution')
plt.tight_layout()
plt.savefig('output/poverty_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to output/poverty_distribution.png')

In [ ]:
# ── Mean dimension scores ────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

dim_cols = ['Availability_score', 'Reliability_score', 'Adequacy_score',
            'Quality_score', 'Affordability_score']
dim_means = mepi_results[dim_cols].mean()
dim_labels = [c.replace('_score', '') for c in dim_cols]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(dim_labels, dim_means.values,
              color=['#3498db', '#e74c3c', '#2ecc71', '#9b59b6', '#f39c12'])
ax.axhline(0.33, color='orange', linestyle='--', linewidth=1.2, label='Deprivation threshold (0.33)')
ax.axhline(0.66, color='red',    linestyle='--', linewidth=1.2, label='Severe threshold (0.66)')
ax.set_ylim(0, 1)
ax.set_ylabel('Mean Deprivation Score')
ax.set_title('Mean MEPI Dimension Scores')
ax.legend(fontsize=9)
for bar, val in zip(bars, dim_means.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('output/dimension_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to output/dimension_scores.png')

In [ ]:
# ── Download output files to your local machine ──────────────────────────────
import os
import glob
from google.colab import files

DOWNLOAD_FILES = True  # Set to False to skip automatic download

if DOWNLOAD_FILES:
    output_files = glob.glob('output/*') + glob.glob(os.path.expanduser('~/energy_poverty_reports/*'))
    downloadable = [f for f in output_files if os.path.isfile(f)]

    if downloadable:
        print(f'Downloading {len(downloadable)} file(s) …')
        for f in sorted(downloadable):
            size_kb = os.path.getsize(f) / 1024
            print(f'  ⬇️  {f}  ({size_kb:.1f} KB)')
            files.download(f)
        print('\n✅ Downloads initiated.')
    else:
        print('No output files found. Run the analysis sections first.')
else:
    print('ℹ️  Download skipped. Set DOWNLOAD_FILES = True to download.')

---
## 5. Custom Configuration <a id='custom-config'></a>

Explore advanced features: custom dimension weights, geographic zone analysis, and sensitivity analysis.

### 5a. Custom Dimension Weights

Override the default equal-weighting scheme.  
Weights must sum to **1.0**.

In [ ]:
# ── Define and apply custom weights ──────────────────────────────────────────
from mepi_calculator import MEPICalculator

# Example: emphasise Availability and Affordability
CUSTOM_WEIGHTS = {
    'Availability':   0.30,
    'Reliability':    0.15,
    'Adequacy':       0.20,
    'Quality':        0.10,
    'Affordability':  0.25,
}
assert abs(sum(CUSTOM_WEIGHTS.values()) - 1.0) < 1e-9, 'Weights must sum to 1.0'

calc_custom = MEPICalculator(weights=CUSTOM_WEIGHTS)
results_custom = calc_custom.calculate(df_clean)

print('Custom weights applied:')
for dim, w in CUSTOM_WEIGHTS.items():
    print(f'  {dim:15s}: {w:.2f}')

print(f'\nMean MEPI (custom weights) : {results_custom["mepi_score"].mean():.4f}')
print(f'Mean MEPI (equal weights)  : {mepi_results["mepi_score"].mean():.4f}')

# Compare top 5
print('\n── Top 5 (custom weights) ──')
display(results_custom[['upazila_name', 'district', 'mepi_score', 'poverty_category']]
        .sort_values('mepi_score', ascending=False).head(5).round(4))

### 5b. Geographic Zone Analysis

Aggregate MEPI results by Bangladesh's administrative divisions and special ecological zones (coastal, char, haor, hill tract, Sundarbans, plain).

In [ ]:
# ── Zone analysis ────────────────────────────────────────────────────────────
from data_utils import assign_geographic_zone
from analysis import (
    aggregate_by_district,
    aggregate_by_division,
    aggregate_by_zone,
    rank_upazilas,
)

# Assign geographic zone labels
results_with_zone = assign_geographic_zone(mepi_results.copy())

# District-level aggregation
district_agg = aggregate_by_district(mepi_results)
print('── Mean MEPI by District (top 10) ──')
display(district_agg.sort_values('mepi_score', ascending=False).head(10).round(4))

# Division-level aggregation
division_agg = aggregate_by_division(mepi_results)
print('\n── Mean MEPI by Division ──')
display(division_agg.sort_values('mepi_score', ascending=False).round(4))

# Zone-level aggregation
zone_agg = aggregate_by_zone(results_with_zone)
print('\n── Mean MEPI by Geographic Zone ──')
display(zone_agg.round(4))

In [ ]:
# ── Division-level bar chart ──────────────────────────────────────────────────
import matplotlib.pyplot as plt

div_sorted = division_agg.sort_values('mepi_score', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#e74c3c' if v > 0.5 else '#f39c12' if v > 0.33 else '#2ecc71'
          for v in div_sorted['mepi_score']]
bars = ax.barh(div_sorted.index, div_sorted['mepi_score'], color=colors)
ax.axvline(0.33, color='orange', linestyle='--', linewidth=1.2, label='Moderate threshold')
ax.axvline(0.66, color='red',    linestyle='--', linewidth=1.2, label='Severe threshold')
ax.set_xlabel('Mean MEPI Score')
ax.set_title('Mean MEPI Score by Division')
ax.legend(fontsize=9)
for bar, val in zip(bars, div_sorted['mepi_score']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('output/division_mepi.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to output/division_mepi.png')

### 5c. Sensitivity Analysis

Compare MEPI scores under different weighting schemes to assess how robust the rankings are.

In [ ]:
# ── Sensitivity analysis ──────────────────────────────────────────────────────
from config import DEFAULT_WEIGHTS, DIMENSION_WEIGHTS_ALT1, DIMENSION_WEIGHTS_ALT2
from analysis import sensitivity_comparison_table

weight_schemes = [DEFAULT_WEIGHTS, DIMENSION_WEIGHTS_ALT1, DIMENSION_WEIGHTS_ALT2]
scheme_labels  = ['Equal', 'Availability-Affordability', 'Reliability-Adequacy']

print('Running sensitivity analysis across 3 weight schemes …')
all_results = calc.calculate_with_sensitivity(df_clean, weight_schemes)

# Build comparison table
comparison = sensitivity_comparison_table(
    df_clean,
    weight_schemes,
    scheme_labels,
)
print('\n── Sensitivity Comparison (top 10 upazilas by equal-weight MEPI) ──')
display(comparison.head(10).round(4))

In [ ]:
# ── Sensitivity scatter plot ──────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

r0 = all_results[0]['mepi_score'].values
r1 = all_results[1]['mepi_score'].values
r2 = all_results[2]['mepi_score'].values

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, r, label in zip(axes, [r1, r2],
                        ['Availability-Affordability', 'Reliability-Adequacy']):
    ax.scatter(r0, r, alpha=0.6, s=30, color='steelblue')
    lim = [0, 1]
    ax.plot(lim, lim, 'r--', linewidth=1, label='x = y')
    corr = np.corrcoef(r0, r)[0, 1]
    ax.set_xlabel('Equal Weights MEPI')
    ax.set_ylabel(f'{label} MEPI')
    ax.set_title(f'Sensitivity: Equal vs {label}\nr = {corr:.3f}')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('output/sensitivity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to output/sensitivity_analysis.png')

---
## ✅ All Done!

You have successfully:
- Installed all dependencies
- Loaded and validated energy poverty data
- Calculated MEPI scores
- Generated visualizations and spatial maps
- Exported results to CSV and Excel
- Run a sensitivity analysis with alternative weights

### Next Steps
- Upload your own survey data (Section 3) and rerun the analysis
- Adjust the weights in Section 5a to match your research priorities
- Download reports and share with stakeholders (Section 4)
- Explore the repository scripts for more advanced functionality:
  - `generate_all_maps_organized.py` – full map suite
  - `visualization_example.py` – comprehensive chart gallery
  - `generate_full_report.py` – PDF + DOCX report

For questions, open an issue at [github.com/iqbal1999-tiger/energy-poverty-mapping](https://github.com/iqbal1999-tiger/energy-poverty-mapping).